# Build & Evaluate a RAG Pipeline with Qdrant
Dataset: `rag-datasets/rag-mini-wikipedia` · LLM: Gemini · Vector store: Qdrant (in-memory) · All sampling uses `seed=42`.

**Note:** `text-embedding-004` and `gemini-2.0-flash` were retired by Google, so this notebook uses `gemini-embedding-001` (768-dim) and `gemini-2.5-flash`. Free-tier rate limits are handled with batching + waits (a few extra minutes).

In [1]:
!pip install -q "langchain>=0.3,<0.4" "langchain-core>=0.3,<0.4" "langchain-text-splitters>=0.3,<0.4" "langchain-google-genai>=2.0" "langchain-qdrant>=0.2,<0.3" "qdrant-client>=1.12" "datasets>=3.0,<4" "pandas>=2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 447.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not c

In [3]:
import os
from getpass import getpass
SEED = 42
os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key: ")

Paste your Gemini API key: ··········


## Phase 1 — Load corpus (200-passage sample, seed=42)

In [4]:
from datasets import load_dataset
from langchain_core.documents import Document

corpus = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus")["passages"]
print("Full corpus size:", len(corpus))
sample = corpus.shuffle(seed=SEED).select(range(200))
print("Sample size:", len(sample))
docs = [Document(page_content=r["passage"], metadata={"id": r["id"]}) for r in sample]

README.md:   0%|          | 0.00/719 [00:00<?, ?B/s]

data/passages.parquet/part.0.parquet: reconstructing file:   0%|          |  0.00B /  797kB            

data/passages.parquet/part.0.parquet: downloading bytes:           |  0.00B            

Generating passages split:   0%|          | 0/3200 [00:00<?, ? examples/s]

Full corpus size: 3200
Sample size: 200


## Phase 2 — Chunk + Embed

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print("Chunks after splitting:", len(chunks))

from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    output_dimensionality=768,
)
vec = embeddings.embed_query(chunks[0].page_content)
print("Embedding dimension:", len(vec))

Chunks after splitting: 283
Embedding dimension: 3072


## Phase 3 — Qdrant (in-memory) collection `mini_wikipedia`
Upserted in batches with waits to respect the free-tier limit (100 embed requests/min).

In [6]:
import time
from langchain_qdrant import QdrantVectorStore

BATCH = 80
vectorstore = None
for i in range(0, len(chunks), BATCH):
    batch = chunks[i:i + BATCH]
    for attempt in range(6):
        try:
            if vectorstore is None:
                vectorstore = QdrantVectorStore.from_documents(
                    batch, embeddings, location=":memory:", collection_name="mini_wikipedia")
            else:
                vectorstore.add_documents(batch)
            break
        except Exception as e:
            print("Rate limit hit, waiting 65s...", str(e)[:60])
            time.sleep(65)
    print(f"Upserted {min(i + BATCH, len(chunks))}/{len(chunks)}")
    if i + BATCH < len(chunks):
        time.sleep(65)

print(vectorstore.client.get_collection("mini_wikipedia").points_count, "points in collection")
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

Upserted 80/283
Upserted 160/283
Upserted 240/283
Upserted 283/283
283 points in collection


## Phase 4 — RAG chain (LCEL)

In [7]:
import requests
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

prompt = ChatPromptTemplate.from_template(
    "Answer ONLY from the context below in 1-2 short sentences. "
    "If the answer isn't there, say 'I don't know'.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

def make_chain(model_name):
    llm = ChatGoogleGenerativeAI(model=model_name, temperature=0, max_retries=0)
    return ({"context": retriever | format_docs, "question": RunnablePassthrough()}
            | prompt | llm | StrOutputParser())

def ask(question):
    for _ in range(3):
        try:
            return chain.invoke(question)
        except Exception as e:
            msg = str(e)
            if "PerDay" in msg or "404" in msg or "NotFound" in type(e).__name__:
                raise  # daily quota / retired model: retrying won't help
            print("Per-minute limit, waiting 20s...")
            time.sleep(20)
    raise RuntimeError("Failed after retries")

# List the Gemini models this key can actually use (free-tier daily quotas are per model)
resp = requests.get("https://generativelanguage.googleapis.com/v1beta/models",
                    params={"key": os.environ["GOOGLE_API_KEY"], "pageSize": 200}).json()
names = [m["name"].split("/")[-1] for m in resp.get("models", [])
         if "generateContent" in m.get("supportedGenerationMethods", [])]
flash = [n for n in names if "flash" in n]
stable = [n for n in flash if not any(x in n for x in ["image", "tts", "live", "audio", "exp", "robotics", "computer"])]
cands = sorted(stable or flash, key=lambda n: ("lite" not in n, "preview" in n, n))  # lite first: bigger free quota
print("Candidate models:", cands)

q = "Was Abraham Lincoln the sixteenth President of the United States?"
print("Question:", q)
for i, d in enumerate(retriever.invoke(q), 1):
    print(f"\n--- chunk {i} ---\n{d.page_content}")

MODEL = None
for m in cands:
    chain = make_chain(m)
    try:
        answer = ask(q)
        MODEL = m
        break
    except Exception as e:
        print("Skipping", m, "->", str(e)[:70].replace("\n", " "))
assert MODEL, "No model has free quota left right now. Wait for the daily reset or use a key from a new AI Studio project."
print("\nUsing LLM:", MODEL)
print("ANSWER:", answer)

Candidate models: ['gemini-2.5-flash-lite', 'gemini-3.1-flash-lite', 'gemini-3.5-flash-lite', 'gemini-flash-lite-latest', 'gemini-3.1-flash-lite-preview', 'gemini-2.5-flash', 'gemini-3.5-flash', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'gemini-flash-latest', 'gemini-omni-1.1-flash', 'gemini-3-flash-preview', 'gemini-omni-flash-preview']
Question: Was Abraham Lincoln the sixteenth President of the United States?

--- chunk 1 ---
Abraham Lincoln was born on February 12, 1809, to Thomas Lincoln and Nancy Hanks, two uneducated farmers. Lincoln was born in a one-room log cabin on the   Sinking Spring Farm, in southeast Hardin County, Kentucky (now part of LaRue County). This area was at the time considered the "frontier." The name "Abraham" was chosen to commemorate his grandfather, who was killed in an American Indian raid in 1786. Donald (1995) p 21  His elder sister, Sarah Lincoln, was born in 1807; a younger brother,

--- chunk 2 ---
Ulysses S. Grant, See military car

## Phase 5 — Evaluate on 20 real questions (seed=42)
`correct = 1` if lowercase ground truth is a substring of lowercase prediction. The metric is fixed and not adjusted after seeing results.

In [8]:
import pandas as pd
qa = load_dataset("rag-datasets/rag-mini-wikipedia", "question-answer")["test"]
eval_set = qa.shuffle(seed=SEED).select(range(20))

rows = []
for r in eval_set:
    question, gt = r["question"], str(r["answer"])
    retrieved = retriever.invoke(question)
    pred = ask(question)
    correct = int(gt.lower() in pred.lower())
    rows.append({
        "question": question,
        "retrieved_chunks": " || ".join(d.page_content[:200] for d in retrieved),
        "prediction": pred,
        "ground_truth": gt,
        "correct": correct,
    })
    print(f"Q: {question}\nChunks: {[d.page_content[:200] for d in retrieved]}\nPred: {pred}\nGT: {gt}\nCorrect: {correct}\n")
    time.sleep(8)

df = pd.DataFrame(rows)
df.to_csv("eval_results.csv", index=False)
accuracy = df["correct"].sum() / 20
print(f"ACCURACY: {accuracy:.2f} ({df['correct'].sum()}/20)")

data/test.parquet/part.0.parquet: reconstructing file:   0%|          |  0.00B / 54.4kB            

data/test.parquet/part.0.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/918 [00:00<?, ? examples/s]

Q: What did Cleveland's opponents say in 1884 to counter his innocent image?
Chunks: ["* Cleveland, Grover.  about Hawaii.'' (1893).", 'Cleveland was born in Caldwell, New Jersey to the Reverend Richard Cleveland and Anne Neal.  He was the fifth of nine children, five sons and four daughters.  He was named Stephen Grover in honor of t', 'Wilson had some harsh words to say about immigrants in his history books. However, after he entered politics in 1910, Wilson worked to integrate new immigrants into the Democratic party, into the army', 'Invoking the Monroe Doctrine in 1895, Cleveland forced Britain to agree to arbitration of a disputed boundary in Venezuela. His administration is credited with the modernization of the United States N']
Pred: I don't know.
GT: That he had fathered an illegitimate child
Correct: 0

Q: Does otter give birth or lay egg?
Chunks: ['Eurasian otter', 'Unlike most marine mammals such as (seals or whales), sea otters do not have a layer of insulating blubber. A

In [9]:
# Failed rows, for the README failure analysis; then download the CSV
display(df[df.correct == 0][["question", "prediction", "ground_truth"]])
from google.colab import files
files.download("eval_results.csv")

,question,prediction,ground_truth
0,What did Cleveland's opponents say in 1884 to ...,I don't know.,That he had fathered an illegitimate child
1,Does otter give birth or lay egg?,I don't know.,give birth
2,How many days did it take the Imperial Japanes...,I don't know.,Six days
3,"The John Adams Library , housed at the Boston ...",I don't know.,"Adams's personal collection of more than 3,500..."
4,Who is the most popular rock group in Finland?,I don't know.,CMX
5,Was Wilson president of the American Political...,I don't know.,Yes
6,When did the first verifiable written document...,I don't know.,Twelfth Century
7,May a single female lay from several dozen to ...,I don't know.,Yes.
8,Can turtles take many years to reach breeding ...,I don't know.,yes
9,Did Grover Cleveland win the 1884 election?,I don't know.,yes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>